# Defining the baseline CNN model for deep fake classification

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Imports

In [26]:
#Graphics
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

In [27]:
#Keras
from keras.models import Model
from keras import Input, layers, optimizers, callbacks

from keras.utils import image_dataset_from_directory

In [28]:
#Storing location of the data
train_data_dir = "/content/drive/MyDrive/dfake_data_light/train/"
val_data_dir = "/content/drive/MyDrive/dfake_data_light/valid/"
test_data_dir = "/content/drive/MyDrive/dfake_data_light/test/"

In [29]:
BATCH_SIZE = 512
IMAGE_SIZE = (256, 256)
IMAGE_HEIGHT = IMAGE_SIZE[0]
IMAGE_WIDTH = IMAGE_SIZE[1]
NUM_CHANNELS = 3
SEED = 42

In [31]:
train_ds = image_dataset_from_directory(
    train_data_dir,
    labels="inferred",
    label_mode="binary",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE)


val_ds = image_dataset_from_directory(
    val_data_dir,
    labels="inferred",
    label_mode="binary",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE)

test_ds = image_dataset_from_directory(
    test_data_dir,
    labels="inferred",
    label_mode="binary",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE)

Found 10000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.


In [39]:
class_names = train_ds.class_names
print(class_names)

['fake', 'real']


In [33]:
def initialize_model():
    #Input
    inputs = Input(shape=(IMAGE_HEIGHT, IMAGE_WIDTH, NUM_CHANNELS))

    #Normalization
    x = layers.Rescaling(1./255)(inputs)

    #CNN
    x = layers.Conv2D(filters=32, kernel_size=(3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(pool_size=(2,2), padding='same')(x)

    x = layers.Conv2D(filters=64, kernel_size=(3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(pool_size=(2,2), padding='same')(x)

    x = layers.Conv2D(filters=128, kernel_size=(3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(pool_size=(2,2), padding='same')(x)

    # x = layers.Conv2D(filters=256, kernel_size=(3,3), activation='relu', padding='same')(x)
    # x = layers.MaxPooling2D(pool_size=(2,2), padding='same')(x)

    x = layers.Flatten()(x)

    #Dense layers
    # x = layers.Dense(256, activation='relu')(x)
    # x = layers.Dense(128, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dense(32, activation='relu')(x)

    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)

    return model

In [34]:
model = initialize_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_1 (Rescaling)         │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 131072)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │     8,388,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,484,033 (32.36 MB)

 Trainable params: 8,484,033 (32.36 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
# adam = optimizers.Adam(learning_rate=0.001)

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [36]:
MODEL = 'model.keras'

modelCheckpoint = callbacks.ModelCheckpoint(MODEL,
                                            monitor="val_loss",
                                            verbose=0,
                                            save_best_only=True)

# LRreducer = callbacks.ReduceLROnPlateau(monitor="val_loss",
#                                         factor=0.1,
#                                         patience=3,
#                                         verbose=1,
#                                         min_lr=0)

EarlyStopper = callbacks.EarlyStopping(monitor='val_loss',
                                       patience=10,
                                       verbose=0,
                                       restore_best_weights=True)

In [37]:
%%time
history = model.fit(train_ds,
                    epochs=1,
                    validation_data=val_ds,
                    callbacks=[modelCheckpoint, EarlyStopper],
                    verbose=1)

40/40 ━━━━━━━━━━━━━━━━━━━━ 2358s 57s/step - accuracy: 0.5148 - loss: 0.8563 - val_accuracy: 0.5000 - val_loss: 0.7085
CPU times: user 44.6 s, sys: 4.09 s, total: 48.7 s
Wall time: 39min 18s


In [ ]:
def plot_history(history):
    fig, ax = plt.subplots(1, 2, figsize=(15,5))
    ax[0].set_title('loss')
    ax[0].plot(history.epoch, history.history["loss"], label="Train loss")
    ax[0].plot(history.epoch, history.history["val_loss"], label="Validation loss")
    ax[1].set_title('accuracy')
    ax[1].plot(history.epoch, history.history["accuracy"], label="Train acc")
    ax[1].plot(history.epoch, history.history["val_accuracy"], label="Validation acc")
    ax[0].legend()
    ax[1].legend()

In [ ]:
# plot_history(history)

In [40]:
y_pred = model.predict(test_ds)
y_pred

8/8 ━━━━━━━━━━━━━━━━━━━━ 493s 61s/step


array([[0.6711579 ],
       [0.6042144 ],
       [0.67578   ],
       ...,
       [0.6522644 ],
       [0.63935745],
       [0.61586124]], dtype=float32)

In [ ]:
import joblib

# model is your trained model
joblib.dump(model, "/content/drive/MyDrive/dfake_models/baseline_v0.joblib")

['/content/drive/MyDrive/dfake_models/baseline_v0.joblib']